In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the Excel file
file_path = "input C13 data.xlsx"
sheet_name = "Data more peaks"

# Load the mean values table (from U18 to AF48) and errors table (from U53 to AF83)
mean_values = pd.read_excel(file_path, sheet_name=sheet_name, usecols="S:AD", skiprows=9, nrows=31)
errors = pd.read_excel(file_path, sheet_name=sheet_name, usecols="S:AD", skiprows=44, nrows=31)

# Strip any leading/trailing spaces in column names (only need to do this once)
mean_values.columns = mean_values.columns.str.strip()
errors.columns = mean_values.columns  # Reuse the same column names for the errors DataFrame

# Extract the 'time' column
time = mean_values['time']

# Extract metabolite names (all columns except 'time')
metabolites = mean_values.columns[1:]

# Customize plot settings here
x_limits = (-1, 140)  # X-axis range, change as needed
y_limits = (-5, 10)  # Adjust Y-axis range as needed to display meaningful negative values
x_label = "Time (minutes)"
y_label = "Mean Value"
common_title = "Metabolite Parameter vs Time"

# 1) Display individual scatter plots with error bars for each metabolite
for metabolite in metabolites:
    plt.figure(figsize=(8, 6))
    # Use absolute values for the error margins to avoid the error
    yerr_abs = errors[metabolite].abs()
    plt.errorbar(time, mean_values[metabolite], yerr=yerr_abs, fmt='o', capsize=5, label=metabolite)
    
    # Set title and axis labels
    plt.title(f'{metabolite} - {common_title}')
    plt.xlabel(x_label)
    plt.ylabel(f'{metabolite} {y_label}')
    
    # Set axis limits
    plt.xlim(x_limits)
    plt.ylim(y_limits)
    
    plt.legend()
    plt.grid(True)
    
    # Show the plot
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import matplotlib.patches as patches

plt.figure(figsize=(8, 6))

# Customize plot settings here
x_limits = (-1, 140)  # X-axis range
y_limits = (-0.1, 10)  # Y-axis range

x_limits_inset = (-1, 21)  # X-axis range for inset
y_limits_inset = (-0.1, 6)  # Y-axis range for inset

# Plot each metabolite except the last one
for metabolite in metabolites[:-1]:
    plot_label = metabolite.replace("_", "-").rstrip(".1")
    yerr_abs = errors[metabolite].abs()  # Ensure no negative error values
    plt.errorbar(time, mean_values[metabolite], yerr=yerr_abs, fmt='o', capsize=5, label=plot_label)

# Plot the last metabolite in black
last_metabolite = metabolites[-1]
plot_label = last_metabolite.replace("_", "-").rstrip(".1")
yerr_abs_last = errors[last_metabolite].abs()  # Ensure no negative error values
plt.errorbar(time, mean_values[last_metabolite], yerr=yerr_abs_last, fmt='o', capsize=5, color='k', label=plot_label)

# Axis labels
plt.xlabel('time (min)', fontsize=16)
plt.ylabel('$^{13}$C content (% $^{13}$C atoms in the pool)', fontsize=16)

# Set font size for tick labels on the main plot
plt.tick_params(axis='x', labelsize=16)  # X-axis tick labels font size
plt.tick_params(axis='y', labelsize=16)  # Y-axis tick labels font size

# Set axis limits
plt.xlim(x_limits)
plt.ylim(y_limits)

# Remove top and right spines
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Place the legend below the plot
plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=4, title="")

# Highlight the inset area on the main plot with a dashed rectangle
rect = patches.Rectangle((x_limits_inset[0], y_limits_inset[0]),  # Lower-left corner
                         x_limits_inset[1] - x_limits_inset[0],  # Width
                         y_limits_inset[1] - y_limits_inset[0],  # Height
                         linewidth=1.5, edgecolor='gray', linestyle='--', facecolor='none')
ax.add_patch(rect)

# Create inset axes
ax_inset = inset_axes(ax, width="30%", height="30%", loc='upper center')  # Adjust size and position as needed

# Plot each metabolite in the inset, except the last one
for metabolite in metabolites[:-1]:
    plot_label = metabolite.replace("_", "-").rstrip(".1")
    yerr_abs = errors[metabolite].abs()  # Ensure no negative error values
    ax_inset.errorbar(time, mean_values[metabolite], yerr=yerr_abs, fmt='o', capsize=5)

# Plot the last metabolite in the inset in black
yerr_abs_last = errors[last_metabolite].abs()  # Ensure no negative error values
ax_inset.errorbar(time, mean_values[last_metabolite], yerr=yerr_abs_last, fmt='o', capsize=5, color='k')

# Set inset axis limits
ax_inset.set_xlim(x_limits_inset)
ax_inset.set_ylim(y_limits_inset)

# Set font size for tick labels on the inset plot
ax_inset.tick_params(axis='x', labelsize=12)
ax_inset.tick_params(axis='y', labelsize=12)

# Optionally, add grid and labels to the inset (if needed)
ax_inset.grid(False)

# Save combined plot as TIFF with 150 dpi
plt.savefig("all_metabolites_scatter_plot_with_inset.tiff", dpi=150, format='tiff', bbox_inches='tight')

# Show the plot
plt.show()


In [ ]:
# Create a new DataFrame with the required columns
df_phosphoglycerate = pd.DataFrame({
    "t_experimental": time,
    "mean": mean_values["phosphoglycerate"],
    "SD": errors["phosphoglycerate"]
})

# Save the DataFrame to a CSV file without the indexing column
csv_file_path = "P3G_labeling_data.csv"
df_phosphoglycerate.to_csv(csv_file_path, index=False)

print(f"CSV file '{csv_file_path}' has been created successfully.")
